In [1]:
import os
from dotenv import load_dotenv, find_dotenv
import requests
from langchain.utilities import SerpAPIWrapper
from langchain.tools import Tool, tool
from langchain.agents import initialize_agent, AgentType, AgentExecutor
from langchain.memory import ConversationBufferMemory
from langchain_google_genai import ChatGoogleGenerativeAI


c:\Users\NABEEL\.conda\envs\GenAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = ChatGoogleGenerativeAI(
    model='gemini-2.0-flash-lite',
    temperature=0.7,
    max_tokens=None,
    timeout=None,
    max_retries=3,
)

In [3]:
load_dotenv(find_dotenv())
def get_api_key(api_key_name):
    return os.getenv(api_key_name)

In [10]:
#print(get_api_key("News_API"))

### defining all the required tools


In [13]:
@tool
def google_search_tool(query:str):
    '''
    Useful for answering questions by searching Google.
    '''
    search = SerpAPIWrapper(serpapi_api_key=get_api_key("SERPAPI_API_KEY"))
    return search.run(query)


@tool
def weather_info_tool(location:str):
    '''
    Useful for gettnig the temperature and other weather realted information 
    '''
    base_url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": location,
        "appid": get_api_key("OPENWEATHERMAP_API_KEY"),
        "units": "metric"  # for temperature in Celsius
    }
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        return f"Error fetching weather: {response.text}"
    
    data = response.json()
    temp = data["main"]["temp"]
    description = data["weather"][0]["description"]
    return f"The weather in {city_name} is {description} with a temperature of {temp}°C."


@tool
def convert_c_to_f(temp_input: str) -> str:
    """
    Converts a temperature from Celsius to Fahrenheit.
    Input should be a string that includes the Celsius value (e.g., '15', '15°C', or full sentence).
    """
    # Extract the first number from the input string
    match = re.search(r'-?\d+(\.\d+)?', temp_input)
    if not match:
        return "Could not find a valid Celsius temperature in the input."
    
    temp_celsius = float(match.group())
    temp_fahrenheit = (temp_celsius * 9/5) + 32
    return f"{temp_celsius}°C is equal to {temp_fahrenheit:.2f}°F."


@tool
def live_cricket_score(input:str):
    '''
    Useful to get the information about on going live cricker score
    Input can be anything (ignored).
    '''
    api_key = get_api_key("Cricket_API")
    #print(api_key)
    url = f"https://api.cricapi.com/v1/currentMatches?apikey={api_key}&offset=0"

    response = requests.get(url)
    if response.status_code != 200:
        return f"Error fetching score: {response.text}"

    data = response.json()
    print(data)

    if not data.get("data"):
        return "No live matches found."

    results = []
    for match in data["data"]:
        if match["status"] == "live":
            team1 = match["teams"][0]
            team2 = match["teams"][1]
            score = match.get("score", [])
            status = match.get("status", "Unknown")

            results.append(f"{team1} vs {team2} - Status: {status}")

    if not results:
        return "No ongoing live matches at the moment."

    return "\n".join(results)

In [ ]:
tools = [google_search_tool,weather_info_tool,convert_c_to_f,live_cricket_score]

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# === Agent with Memory ===
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    #verbose=True,
    handle_parsing_errors=True
)

def chatbot_response(prompt):
    response = agent.invoke(prompt)
    return response.get("output") if isinstance(response, dict) else str(response)



In [ ]:
# while True:
#     query = input("You: ")  # <-- THIS was missing
#     if query.lower() in ["exit", "quit"]:
#         break
#     print("Bot:", chatbot_response(query))


In [ ]:
from IPython.display import display
import ipywidgets as widgets

# Text input widget
input_box = widgets.Text(
    placeholder='Ask me something...',
    description='You:',
    disabled=False
)

# Output display
output = widgets.Output()

# Handler function
def on_enter(change):
    with output:
        user_input = change['new']
        if user_input.lower() in ['exit', 'quit']:
            print("Session ended.")
            input_box.disabled = True
        else:
            response = chatbot_response(user_input)
            print(f"You: {user_input}")
            print(f"Bot: {response}")
        input_box.value = ''  # Clear input after submission

# Bind enter key to function
input_box.observe(on_enter, names='value')

# Display input box and output
display(input_box, output)


Text(value='', description='You:', placeholder='Ask me something...')

Output()